In [ ]:
MODEL_PATH         = "/content/drive/MyDrive/efficientnet.keras" 
NGROK_AUTHTOKEN    = ""       
PORT               = 8000
IMG_SIZE           = 300      
BATCH_SIZE         = 32     
LABELS             = ["nsfw", "safe", "violence"]
NSFW_THRESHOLD     = 0.6
VIOLENCE_THRESHOLD = 0.6
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

import numpy as _np_check
_np_major = int(_np_check.__version__.split(".")[0])
if _np_major >= 2:
    print(f"⚠️  Phát hiện numpy {_np_check.__version__} (TensorFlow 2.17 cần numpy<2).")
    print("🔧 Đang downgrade numpy==1.26.4 và restart kernel...")
    _pip("--force-reinstall", "numpy==1.26.4")
    print("✅ numpy đã được cài lại. Kernel sẽ tự restart sau 2 giây...")
    import time; time.sleep(2)
    os.kill(os.getpid(), 9)

print(f"✅ numpy {_np_check.__version__} — OK, tiếp tục cài đặt...")

print("📦 Đang cài đặt thư viện...")
_pip(
    "fastapi==0.115.5",
    "uvicorn[standard]==0.32.1",
    "python-multipart==0.0.18",
    "tensorflow==2.17.0",
    "Pillow==11.0.0",
    "pyngrok",
    "nest_asyncio",
)
print("✅ Cài đặt xong!")

from google.colab import drive
drive.mount("/content/drive")

import logging
from typing import Annotated

import numpy as np
import tensorflow as tf
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
)
logger = logging.getLogger("image_moderation")

class FramePrediction(BaseModel):
    frame:      str
    label:      str
    confidence: float = Field(..., ge=0.0, le=1.0)
    scores:     dict[str, float]

class BatchPredictResponse(BaseModel):
    total:       int
    predictions: list[FramePrediction]

class HealthResponse(BaseModel):
    status:       str
    model_loaded: bool
    labels:       list[str]

_model: tf.keras.Model | None = None

def load_model() -> tf.keras.Model:
    global _model
    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(
            f"Không tìm thấy model: {MODEL_PATH}\n"
            "Hãy upload efficientnet.keras lên Google Drive và sửa MODEL_PATH."
        )
    logger.info(f"🖼️  Loading EfficientNet from: {MODEL_PATH}")
    _model = tf.keras.models.load_model(MODEL_PATH)
    gpus = tf.config.list_physical_devices("GPU")
    logger.info(f"✅ Model loaded — GPU available: {len(gpus) > 0} ({len(gpus)} device(s))")
    return _model

def preprocess_image(image_bytes: bytes) -> np.ndarray:
    img = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)  
    return img.numpy()

def predict_batch(images: list[np.ndarray]) -> list[dict]:
    batch = np.stack(images, axis=0)
    raw_preds: np.ndarray = _model.predict(batch, verbose=0)
    results = []
    for pred in raw_preds:
        prob_map = {label: float(pred[i]) for i, label in enumerate(LABELS)}
        if prob_map.get("nsfw", 0.0) > NSFW_THRESHOLD:
            predicted_label = "nsfw"
        elif prob_map.get("violence", 0.0) > VIOLENCE_THRESHOLD:
            predicted_label = "violence"
        else:
            predicted_label = max(prob_map, key=prob_map.get)
        results.append({
            "label":      predicted_label,
            "confidence": prob_map[predicted_label],
            "scores":     prob_map,
        })
    return results

print("\n🚀 Đang load model...")
load_model()
print("✅ Model đã sẵn sàng!\n")

app = FastAPI(
    title="VideoGuard Image Moderation",
    description="EfficientNet: phân loại frame video thành nsfw / safe / violence.",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse, tags=["health"])
def health():
    return HealthResponse(
        status="ok",
        model_loaded=_model is not None,
        labels=LABELS,
    )

@app.post(
    "/images/predict/batch",
    response_model=BatchPredictResponse,
    tags=["predict"],
    summary="Predict labels for a batch of frames",
)
async def predict_frames_batch(
    files: Annotated[list[UploadFile], File(description="Danh sách frame JPEG/PNG")],
):
    if not files:
        raise HTTPException(status_code=422, detail="No files uploaded.")
    if len(files) > BATCH_SIZE:
        raise HTTPException(
            status_code=422,
            detail=f"Too many frames. Max batch size is {BATCH_SIZE}.",
        )

    images, filenames = [], []
    for upload in files:
        raw = await upload.read()
        try:
            arr = preprocess_image(raw)
        except Exception as exc:
            logger.warning(f"Failed to decode {upload.filename}: {exc}")
            raise HTTPException(
                status_code=422,
                detail=f"Cannot decode image: {upload.filename}",
            )
        images.append(arr)
        filenames.append(upload.filename or f"frame_{len(filenames)}.jpg")

    logger.info(f"[predict] Running inference on {len(images)} frames")
    raw_results = predict_batch(images)
    predictions = [
        FramePrediction(frame=fname, **res)
        for fname, res in zip(filenames, raw_results)
    ]
    return BatchPredictResponse(total=len(predictions), predictions=predictions)

from pyngrok import ngrok, conf

if not NGROK_AUTHTOKEN:
    raise ValueError("❌ Hãy điền NGROK_AUTHTOKEN ở phần CẤU HÌNH phía trên!")

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()
public_url = ngrok.connect(PORT, "http").public_url
print(f"\n{'='*60}")
print(f"🌐 Public API URL  : {public_url}")
print(f"📖 Swagger Docs    : {public_url}/docs")
print(f"❤️  Health Check   : {public_url}/health")
print(f"🖼️  Predict Batch  : POST {public_url}/images/predict/batch")
print(f"\n💡 Cấu hình Go backend (.env):")
print(f"   AI_FRAME_MODERATOR_URL={public_url}")
print(f"{'='*60}\n")

uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
